# DHRP: Differentiable Hierarchical Risk Parity

**Full Experimental Pipeline (Colab GPU)**

This notebook reproduces every numerical result reported in the DHRP paper. The pipeline:
1. Loads 14 years of daily price data for three balanced 10-asset ETF universes (developed markets, emerging markets, commodities) and Fama-French factors.
2. Trains the DHRP layer end-to-end on the in-sample period (2012-04 through 2020-06) with multi-objective loss (CRRA + Sharpe + curriculum HRP regularization).
3. Backtests DHRP and a full benchmark set (EW, MV, MINVAR, MAXDIV, HRP, RP, MLP, Transformer, PPO) on the held-out OOS period (2020-07 through 2026-04).
4. Reports a layered statistical battery: HAC-robust Sharpe t-stats, Fama-French six-factor alphas, Jobson-Korkie/Memmel parametric Sharpe tests, stationary block bootstrap with Holm-Bonferroni correction, Diebold-Mariano forecast accuracy, SPA, and Model Confidence Set.
5. Adds ablations (tree depth, loss components, hierarchical inductive bias), multi-seed robustness, transaction-cost sensitivity, and regime-conditional analysis.

**LLM-DHRP** (a multimodal extension fusing FinBERT + Qwen3-8B sentiment via gated cross-attention) is trained and backtested under the same protocol but reported only as a *brief negative ablation*: it does not improve over the base DHRP layer and is mentioned in the paper purely so practitioners do not assume the opposite. The headline contribution is DHRP.

Run all cells in order. Upload your `HRP_AI` folder to Google Drive first, then cell 1 mounts Drive and sets up the environment.


In [1]:
# === CELL 1: SETUP ===
import os, sys, subprocess, glob
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except ValueError:
        pass  # already mounted

    CANDIDATE_PATHS = [
        "/content/drive/MyDrive/HRP_AI",
        "/content/drive/My Drive/HRP_AI",
        "/content/drive/MyDrive/Code/HRP_AI",
        "/content/drive/MyDrive/Projects/HRP_AI",
        "/content/drive/MyDrive/dhrp-allocation",
    ]
    REPO_PATH = None
    for p in CANDIDATE_PATHS:
        if os.path.exists(p):
            REPO_PATH = p
            break

    if REPO_PATH is None:
        REPO_PATH = "/content/HRP_AI"
        if not os.path.exists(REPO_PATH):
            print("HRP_AI not found on Drive - cloning from GitHub...")
            subprocess.check_call([
                "git", "clone",
                "https://github.com/joseamador0898/dhrp-allocation.git",
                REPO_PATH,
            ])
        else:
            # Always pull latest to stay in sync
            print("Pulling latest changes...")
            subprocess.call(["git", "-C", REPO_PATH, "fetch", "--all"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.call(["git", "-C", REPO_PATH, "reset", "--hard", "origin/main"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    os.chdir(REPO_PATH)

    # Copy .env from Drive if available, otherwise prompt upload
    env_candidates = [
        '/content/drive/MyDrive/HRP_AI/.env',
        '/content/drive/MyDrive/.env',
        '/content/drive/My Drive/HRP_AI/.env',
    ]
    if not os.path.exists('.env'):
        for ep in env_candidates:
            if os.path.exists(ep):
                import shutil
                shutil.copy(ep, '.env')
                print(f'Copied .env from {ep}')
                break
    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)
    print(f"Working dir: {os.getcwd()}")

    !pip install -q torch transformers tokenizers huggingface-hub         requests pandas-datareader fredapi cvxpy seaborn python-dotenv         scipy statsmodels bitsandbytes accelerate feedparser datasets yfinance \
        gs-quant
else:
    parent = os.path.abspath("..")
    if parent not in sys.path:
        sys.path.insert(0, parent)

import warnings
warnings.filterwarnings("ignore")

# --- Clean stale results from previous runs ---
def clean_stale_results():
    patterns = [
        "results/*.csv", "results/*.png",
        "results/models/*.pt",
        "results/features/*.npz", "results/features/*.csv",
        "results/full/*.csv",
        "results/figures/*.png",
    ]
    total = 0
    for pat in patterns:
        for f in glob.glob(pat):
            os.remove(f)
            total += 1
    if total:
        print(f"Cleaned {total} stale result files.")

clean_stale_results()

for d in ['results', 'results/figures', 'results/models', 'results/features', 'results/full']:
    os.makedirs(d, exist_ok=True)

import torch
import numpy as np
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
device = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_END = '2020-06-30'

Mounted at /content/drive
HRP_AI not found on Drive - cloning from GitHub...
Working dir: /content/HRP_AI
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.6 MB/s eta 0:00:00
Cleaned 7 stale result files.
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [2]:
# === CELL 2: LOAD PRICE + VOLUME DATA (all 3 universes) ===
from datetime import datetime, timedelta
from src.data.price_loader import load_universe, load_fama_french, load_etf_volume_data, UNIVERSES

END = datetime.now().strftime('%Y-%m-%d')
START = (datetime.now() - timedelta(days=14*365)).strftime('%Y-%m-%d')  # 14yrs (INDA Feb 2012 limit)
print(f'Period: {START} to {END}\n')

print(f'=== DM Universe ({len(UNIVERSES["DM"])} ETFs) ===')
DM_prices = load_universe('DM', START, END)

print(f'\n=== EM Universe ({len(UNIVERSES["EM"])} ETFs) ===')
EM_prices = load_universe('EM', START, END)

print(f'\n=== Commodities Universe ({len(UNIVERSES["Commodities"])} ETFs) ===')
CMD_prices = load_universe('Commodities', START, END)

print('\n=== Fama-French Factors ===')
FF = load_fama_french(START, END)

# Load volume data for enhanced features
print('\n=== Volume Data ===')
try:
    DM_vol = load_etf_volume_data(list(UNIVERSES['DM'].values()), START, END)
    EM_vol = load_etf_volume_data(list(UNIVERSES['EM'].values()), START, END)
    CMD_vol = load_etf_volume_data(list(UNIVERSES['Commodities'].values()), START, END)
    print(f'DM volume: {DM_vol.shape}, EM volume: {EM_vol.shape}, CMD volume: {CMD_vol.shape}')
except Exception as e:
    print(f'Volume data unavailable ({e}), proceeding without')
    DM_vol = EM_vol = CMD_vol = None

Period: 2012-04-22 to 2026-04-19

=== DM Universe (10 ETFs) ===
  [1/10] SPY: 3517 days
  [2/10] QQQ: 3517 days
  [3/10] IWM: 3517 days
  [4/10] EFA: 3517 days
  [5/10] VGK: 3517 days
  [6/10] TLT: 3517 days
  [7/10] IEF: 3517 days
  [8/10] LQD: 3517 days
  [9/10] VNQ: 3517 days
  [10/10] UUP: 3517 days
  Total: 3517x10 assets, coverage: 100.0%

=== EM Universe (10 ETFs) ===
  [1/10] EEM: 3517 days
  [2/10] EWZ: 3517 days
  [3/10] FXI: 3517 days
  [4/10] EWY: 3517 days
  [5/10] EWT: 3517 days
  [6/10] INDA: 3517 days
  [7/10] EWW: 3517 days
  [8/10] EZA: 3517 days
  [9/10] THD: 3517 days
  [10/10] TUR: 3517 days
  Total: 3517x10 assets, coverage: 100.0%

=== Commodities Universe (10 ETFs) ===
  [1/10] USO: 3517 days
  [2/10] UNG: 3517 days
  [3/10] GLD: 3517 days
  [4/10] SLV: 3517 days
  [5/10] DBA: 3517 days
  [6/10] DBC: 3517 days
  [7/10] CPER: 3517 days
  [8/10] WEAT: 3517 days
  [9/10] CORN: 3517 days
  [10/10] SOYB: 3517 days
  Total: 3517x10 assets, coverage: 100.0%

=== Fama-F

In [3]:
# === CELL 3: LOAD HEADLINES (all sources including GDELT) ===
from src.data.text_loader import load_all_headlines
import pandas as pd

# Collect headlines from ALL available sources
# GDELT provides 10-year historical coverage (2016-2026)
# yfinance/RSS provide recent headlines
# PhraseBank/FiQA provide static financial NLP training data
all_tickers = (
    list(UNIVERSES['DM'].values()) +
    list(UNIVERSES['EM'].values()) +
    list(UNIVERSES['Commodities'].values())
)
all_tickers = list(set(all_tickers))

print(f'Fetching headlines for {len(all_tickers)} tickers from all sources...')
headlines_df = load_all_headlines(
    all_tickers, START, END,
    max_headlines=100,
    use_rss=True,
    use_phrasebank=True,
    use_fiqa=True,
    use_gdelt=True,           # GDELT historical headlines (primary source)
    gdelt_max_per_ticker=250,  # 250 per ticker per chunk
    gdelt_chunk_months=12,     # 12-month windows (~15 min vs 100 min)
)
print(f'\nTotal headlines: {len(headlines_df)}')
if not headlines_df.empty:
    print(f'Date range: {headlines_df["date"].min()} to {headlines_df["date"].max()}')
    print(f'Tickers with news: {headlines_df["ticker"].nunique()}')
    print(f'Sources: {headlines_df["source"].value_counts().head(10).to_dict()}')
    # Show coverage by year
    headlines_df['year'] = pd.to_datetime(headlines_df['date']).dt.year
    print(f'Headlines by year:\n{headlines_df["year"].value_counts().sort_index().to_string()}')
    headlines_df = headlines_df.drop(columns=['year'])

Fetching headlines for 30 tickers from all sources...
  Loading GDELT historical headlines...
  GDELT: 0/31 tickers cached (0 headlines)
  Fetching 31 tickers (14 chunks each, 8 parallel workers, ETA ~2 min)...
    [1/31] UNG: 738 headlines | 30 left (~76.1m)
    [2/31] IEF: 1301 headlines | 29 left (~38.1m)
    [3/31] SPY: 764 headlines | 28 left (~25.0m)
    [4/31] TLT: 796 headlines | 27 left (~18.3m)
    [5/31] GLD: 930 headlines | 26 left (~14.5m)
    [6/31] DBA: 1028 headlines | 25 left (~11.9m)
    [7/31] VNQ: 0 headlines | 24 left (~10.4m)
    [8/31] CORN: 561 headlines | 23 left (~9.3m)
    [9/31] EFA: 0 headlines | 22 left (~10.8m)
    [10/31] EZA: 1084 headlines | 21 left (~10.1m)
    [11/31] USO: 0 headlines | 20 left (~9.1m)
    [12/31] INDA: 663 headlines | 19 left (~8.3m)
    [13/31] VGK: 463 headlines | 18 left (~7.4m)
    [14/31] SOYB: 337 headlines | 17 left (~6.7m)
    [15/31] SLV: 400 headlines | 16 left (~6.1m)
    [16/31] DBC: 451 headlines | 15 left (~5.6m)
    [

README.md: 0.00B [00:00, ?B/s]

financial_phrasebank.py: 0.00B [00:00, ?B/s]

Financial PhraseBank unavailable: Dataset scripts are no longer supported, but found financial_phrasebank.py
  Loading FiQA dataset...


train.csv: 0.00B [00:00, ?B/s]

...

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/961 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/102 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

    FiQA: 1213 sentences
  Total unique headlines: 10671

Total headlines: 10671
Date range: 2018-03-27 00:00:00 to 2026-04-19 03:04:52.862749
Tickers with news: 31
Sources: {'FiQA': 1111, 'GDELT:finance.yahoo.com': 241, 'GDELT:seekingalpha.com': 161, 'GDELT:dailypolitical.com': 153, 'GDELT:marketscreener.com': 117, 'GDELT:business-standard.com': 112, 'GDELT:economictimes.indiatimes.com': 107, 'GDELT:msn.com': 98, 'GDELT:hellenicshippingnews.com': 93, 'GDELT:iol.co.za': 85}
Headlines by year:
year
2018    1494
2019    1182
2020    1284
2021    1439
2022     929
2023     377
2024    1022
2025     699
2026    2245


In [4]:
# === CELL 4: FINBERT EMBEDDINGS (T4: ~2 min for 1000 headlines) ===
from src.data.llm_features import get_finbert_embeddings

headline_to_emb = {}
if not headlines_df.empty:
    unique_headlines = headlines_df['headline'].unique().tolist()
    print(f'Extracting FinBERT embeddings for {len(unique_headlines)} unique headlines...')

    finbert_embs = get_finbert_embeddings(unique_headlines, batch_size=64, device=device)
    print(f'FinBERT embeddings shape: {finbert_embs.shape}')

    headline_to_emb = {h: finbert_embs[i] for i, h in enumerate(unique_headlines)}
    print(f'VRAM after FinBERT: {torch.cuda.memory_allocated()/1e9:.2f} GB' if torch.cuda.is_available() else 'CPU mode')
else:
    print('No headlines available. Proceeding with price-only features.')

Extracting FinBERT embeddings for 10671 unique headlines...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

FinBERT embeddings shape: (10671, 768)
VRAM after FinBERT: 0.01 GB


In [5]:
# === CELL 5: GEMINI 3.1 STRUCTURED SENTIMENT (API call, zero GPU) ===
from src.data.gemini_sentiment import extract_sentiment_batch, GEMINI_FEATURE_DIM
import os

# Use Gemini 3.1 Pro for 6-dimensional structured sentiment:
# [sentiment_score, risk_level, regime, confidence, sector_impact, rate_sensitivity]
# Schema-enforced JSON output - no parsing errors, no GPU compute.

# API key from .env or Colab secrets
api_key = os.environ.get('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass
if not api_key:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.environ.get('GOOGLE_API_KEY')

sentiment_by_ticker = {}
if api_key and not headlines_df.empty:
    print('=== GEMINI 3.1 PRO STRUCTURED SENTIMENT ===')
    print('  Model: gemini-3.1-pro-preview')
    print(f'  Output: {GEMINI_FEATURE_DIM}-dim structured JSON per headline')
    print('  Compute: API calls (zero GPU)')
    print()
    sentiment_by_ticker = extract_sentiment_batch(
        headlines_df,
        api_key=api_key,
        model='gemini-3.1-pro-preview',
        max_headlines_per_ticker=15,
        batch_size=10,
        delay_between_calls=0.5,
    )
    n_sent = len(sentiment_by_ticker)
    print(f'  Sentiment extracted for {n_sent} tickers')
    print(f'  Feature dim: {GEMINI_FEATURE_DIM} per ticker')
else:
    if not api_key:
        print('No GOOGLE_API_KEY found. Set it in .env or Colab secrets.')
        print('Get a free key at https://aistudio.google.com/')
    else:
        print('No headlines available for sentiment extraction.')


No GOOGLE_API_KEY found. Set it in .env or Colab secrets.
Get a free key at https://aistudio.google.com/


In [6]:
# === CELL 6: BUILD TEXT FEATURE TENSORS (PIT, no look-ahead) ===
from src.data.feature_engineering import build_dataset, DEFAULT_FDIM
from src.data.gemini_sentiment import build_gemini_text_tensor, GEMINI_FEATURE_DIM
import pandas as pd


def _compute_market_fallback(headlines_df, headline_to_emb):
    market_headlines = headlines_df[headlines_df['ticker'] == 'MARKET']['headline'].tolist()
    embs = [headline_to_emb[h] for h in market_headlines if h in headline_to_emb]
    if embs:
        return np.mean(embs, axis=0).astype(np.float32)
    all_embs = list(headline_to_emb.values())
    if all_embs:
        return np.mean(all_embs, axis=0).astype(np.float32)
    return np.zeros(768, dtype=np.float32)


def build_text_tensor_for_universe(prices, universe_tickers, headline_to_emb,
                                   headlines_df, volume=None, train_end=None):
    """Return (train_tensor, pit_dict).

    train_tensor — (n_samples, n_assets, 768) ndarray for training
    (built from headlines dated < train_end; no look-ahead).
    pit_dict — {Timestamp: (n_assets, 768)} for rolling_backtest. Dates
    before train_end map to the prior embedding, dates at/after map to
    the live embedding. Consumed by the date-keyed lookup in
    backtest._llm_dhrp_weights.
    """
    X, S, R, H = build_dataset(prices, volume=volume, fdim=DEFAULT_FDIM)
    n_samp = X.shape[0]
    n_assets = prices.shape[1]
    ticker_list = list(universe_tickers.values())
    market_fallback = _compute_market_fallback(headlines_df, headline_to_emb)

    hdl_df = headlines_df.copy()
    if 'date' in hdl_df.columns:
        hdl_df['date'] = pd.to_datetime(hdl_df['date'], errors='coerce')
    else:
        hdl_df['date'] = pd.NaT
    cutoff = pd.Timestamp(train_end) if train_end is not None else None

    def _embs_for(ticker, mask):
        rows = hdl_df[mask & (hdl_df['ticker'] == ticker)]['headline'].tolist()
        return [headline_to_emb[h] for h in rows if h in headline_to_emb]

    if cutoff is None:
        prior_mask = hdl_df['date'].isna()
        live_mask = pd.Series(True, index=hdl_df.index)
    else:
        prior_mask = hdl_df['date'].isna() | (hdl_df['date'] < cutoff)
        live_mask = hdl_df['date'].isna() | (hdl_df['date'] >= cutoff)

    prior_embs = np.zeros((n_assets, 768), dtype=np.float32)
    live_embs = np.zeros((n_assets, 768), dtype=np.float32)
    coverage = []
    for j, ticker in enumerate(ticker_list):
        p = _embs_for(ticker, prior_mask)
        l = _embs_for(ticker, live_mask)
        coverage.append(len(p) + len(l))
        prior_embs[j] = np.mean(p, axis=0) if len(p) >= 5 else market_fallback
        live_embs[j] = np.mean(l, axis=0) if len(l) >= 5 else prior_embs[j]

    train_tensor = np.tile(prior_embs, (n_samp, 1, 1))
    pit_dict = {}
    for ts in prices.index:
        ts_pd = pd.Timestamp(ts)
        pit_dict[ts_pd] = live_embs if (cutoff is not None and ts_pd >= cutoff) else prior_embs

    min_c, max_c, mean_c = min(coverage), max(coverage), np.mean(coverage)
    zero_c = sum(1 for c in coverage if c < 5)
    print(f'    Coverage: min={min_c} max={max_c} mean={mean_c:.0f}, '
          f'fallback={zero_c}/{n_assets} assets')
    return train_tensor, pit_dict


print('=== Building FinBERT text feature tensors ===')
text_dm = text_em = text_cmd = None
text_dm_pit = text_em_pit = text_cmd_pit = None
if headline_to_emb:
    text_dm, text_dm_pit = build_text_tensor_for_universe(
        DM_prices, UNIVERSES['DM'], headline_to_emb, headlines_df,
        volume=DM_vol, train_end=TRAIN_END)
    print(f'  DM: {text_dm.shape}')
    text_em, text_em_pit = build_text_tensor_for_universe(
        EM_prices, UNIVERSES['EM'], headline_to_emb, headlines_df,
        volume=EM_vol, train_end=TRAIN_END)
    print(f'  EM: {text_em.shape}')
    text_cmd, text_cmd_pit = build_text_tensor_for_universe(
        CMD_prices, UNIVERSES['Commodities'], headline_to_emb, headlines_df,
        volume=CMD_vol, train_end=TRAIN_END)
    print(f'  CMD: {text_cmd.shape}')

print()
print('=== Building Gemini sentiment feature tensors ===')
gemini_dm = gemini_em = gemini_cmd = None
if sentiment_by_ticker:
    for label, prices, universes, vol_data in [
        ('DM', DM_prices, UNIVERSES['DM'], DM_vol),
        ('EM', EM_prices, UNIVERSES['EM'], EM_vol),
        ('CMD', CMD_prices, UNIVERSES['Commodities'], CMD_vol),
    ]:
        X, S, R, H = build_dataset(prices, volume=vol_data, fdim=DEFAULT_FDIM)
        n_samp = X.shape[0]
        tickers = list(universes.values())
        tensor = build_gemini_text_tensor(sentiment_by_ticker, n_samp, tickers)
        print(f'  {label}: {tensor.shape} (n_samples x {GEMINI_FEATURE_DIM})')
        if label == 'DM':
            gemini_dm = tensor
        elif label == 'EM':
            gemini_em = tensor
        elif label == 'CMD':
            gemini_cmd = tensor

print(f'Gemini feature dim: {GEMINI_FEATURE_DIM}')
print('Ready for LLM-DHRP training with Gemini sentiment.')


=== Building FinBERT text feature tensors ===
    Coverage: min=4 max=1311 mean=325, fallback=1/10 assets
  DM: (652, 10, 768)
    Coverage: min=1 max=1087 mean=179, fallback=3/10 assets
  EM: (652, 10, 768)
    Coverage: min=2 max=1028 mean=446, fallback=1/10 assets
  CMD: (652, 10, 768)

=== Building Gemini sentiment feature tensors ===
Gemini feature dim: 6
Ready for LLM-DHRP training with Gemini sentiment.


In [7]:
# === CELL 7: MACRO FEATURES (FRED full + GS Quant + SPGCI, all universes) ===
from src.data.fred_loader import (
    load_fred_data, make_macro_features, make_commodity_features, make_em_features,
    FRED_SERIES_FULL, FRED_SERIES_COMMODITY, FRED_SERIES_EM,
)
from src.data.universe_config import UNIVERSE_DATA_CONFIG
from dotenv import load_dotenv
load_dotenv()

# Set FRED key directly if .env is not available (e.g. Colab clone)
if not os.environ.get('FRED_API_KEY'):
    os.environ['FRED_API_KEY'] = '139e45f095302fed00e434b1158a4ddc'

print('=== MACRO FEATURES (expanded, all universes) ===')

# FULL FRED macro -- 37 verified series including yields, inflation, labor,
# financial conditions, FX, money supply, real economy indicators
dm_macro = None
print('\n--- Loading FULL FRED macro features (37 series) ---')
fred_df = load_fred_data(START, END, series=FRED_SERIES_FULL)
if not fred_df.empty:
    dm_macro = make_macro_features(fred_df)
    print(f'  FRED macro: {dm_macro.shape} ({dm_macro.shape[1]} features)')
else:
    # Fallback to extended (8 series)
    fred_df = load_fred_data(START, END, extended=True)
    if not fred_df.empty:
        dm_macro = make_macro_features(fred_df)
        print(f'  FRED macro (fallback): {dm_macro.shape}')
    else:
        print('  FRED not available.')

# Commodity-specific FRED series (WTI, Brent, Gold, Copper, Wheat, Corn)
print('\n--- Loading Commodity FRED series ---')
cmd_fred = load_fred_data(START, END, series=FRED_SERIES_COMMODITY)
cmd_macro = None
if not cmd_fred.empty:
    cmd_macro = make_commodity_features(cmd_fred)
    print(f'  Commodity FRED: {cmd_macro.shape}')

# EM-specific FRED series (EM corporate OAS, EM HY OAS)
print('\n--- Loading EM FRED series ---')
em_fred = load_fred_data(START, END, series=FRED_SERIES_EM)
em_macro = None
if not em_fred.empty:
    em_macro = make_em_features(em_fred)
    print(f'  EM FRED: {em_macro.shape}')

# GS Quant institutional data: 13 global indices + 2 FX pairs
# (SPX, VIX, NDX, RTY, MXEF, MXWO, MXEA, BCOMTR, DXY, SX5E, NKY, HSI, SHCOMP)
try:
    from src.data.gsquant_loader import load_gs_data
    gs_df = load_gs_data(START, END)
    if not gs_df.empty:
        if dm_macro is not None:
            dm_macro = pd.concat([dm_macro, gs_df], axis=1, join='outer').ffill().bfill()
        else:
            dm_macro = gs_df
        print(f'  Macro (FRED+GS): {dm_macro.shape}')
except Exception as e:
    print(f'  GS Quant: {e}')

# SPGCI commodity assessments (Platts prices, if subscription allows)
try:
    from src.data.spgci_loader import load_spgci_data
    spgci_df = load_spgci_data(START, END)
    if not spgci_df.empty:
        if cmd_macro is not None:
            cmd_macro = pd.concat([cmd_macro, spgci_df], axis=1, join='outer').ffill().bfill()
        else:
            cmd_macro = spgci_df
        print(f'  Commodity macro (FRED+SPGCI): {cmd_macro.shape}')
except Exception as e:
    print(f'  SPGCI: {e}')

# GDELT aggregate sentiment features (daily tone, media attention)
try:
    from src.data.gdelt_loader import make_gdelt_sentiment_features
    if not headlines_df.empty:
        gdelt_only = headlines_df[headlines_df['source'].str.startswith('GDELT:', na=False)]
        if not gdelt_only.empty:
            gdelt_sent = make_gdelt_sentiment_features(gdelt_only)
            if not gdelt_sent.empty:
                if dm_macro is not None:
                    dm_macro = pd.concat([dm_macro, gdelt_sent], axis=1, join='outer').ffill().bfill()
                print(f'  Added GDELT sentiment: {gdelt_sent.shape[1]} features')
except Exception as e:
    print(f'  GDELT sentiment: {e}')

# Merge universe-specific macro with base macro
if dm_macro is not None and cmd_macro is not None:
    cmd_macro_full = pd.concat([dm_macro, cmd_macro], axis=1, join='outer').ffill().bfill()
    cmd_macro_full = cmd_macro_full.loc[:, ~cmd_macro_full.columns.duplicated()]
    print(f'  Commodities total macro: {cmd_macro_full.shape}')
else:
    cmd_macro_full = dm_macro

if dm_macro is not None and em_macro is not None:
    em_macro_full = pd.concat([dm_macro, em_macro], axis=1, join='outer').ffill().bfill()
    em_macro_full = em_macro_full.loc[:, ~em_macro_full.columns.duplicated()]
    print(f'  EM total macro: {em_macro_full.shape}')
else:
    em_macro_full = dm_macro

# Reindex macro onto full price calendar (ffill/bfill leading gaps). This
# preserves the full 10+ year price history instead of trimming to the
# macro start date (which previously collapsed the OOS window visually).
if dm_macro is not None:
    dm_macro = dm_macro.reindex(DM_prices.index).ffill().bfill()
    print(f'\n  Aligned to price calendar: {DM_prices.index.min().strftime("%Y-%m-%d")} to {DM_prices.index.max().strftime("%Y-%m-%d")} ({len(dm_macro)} days, {dm_macro.shape[1]} features)')
    print(f'  Price spans — DM: {DM_prices.index.min().date()} | EM: {EM_prices.index.min().date()} | CMD: {CMD_prices.index.min().date()}')

print(f'\n=== DATA SUMMARY ===')
print(f'DM macro: {dm_macro.shape if dm_macro is not None else "None"}')
print(f'EM macro: {em_macro_full.shape if "em_macro_full" in dir() and em_macro_full is not None else "None"}')
print(f'CM macro: {cmd_macro_full.shape if "cmd_macro_full" in dir() and cmd_macro_full is not None else "None"}')

=== MACRO FEATURES (expanded, all universes) ===

--- Loading FULL FRED macro features (37 series) ---
  FRED macro: (4411, 37) (37 features)

--- Loading Commodity FRED series ---
  Commodity FRED: (3696, 11)

--- Loading EM FRED series ---
  EM FRED: (795, 7)
Loading GS Quant data...
  No GS Quant data available.
Loading SPGCI data...
Install spgci: pip install spgci
Install spgci: pip install spgci
  No SPGCI data available (may need higher access tier).
  Commodities total macro: (4411, 48)
  EM total macro: (4411, 44)

  Aligned to price calendar: 2012-04-23 to 2026-04-17 (3517 days, 37 features)
  Price spans — DM: 2012-04-23 | EM: 2012-04-23 | CMD: 2012-04-23

=== DATA SUMMARY ===
DM macro: (3517, 37)
EM macro: (4411, 44)
CM macro: (4411, 48)


In [ ]:
# === CELL 8: TRAIN DHRP + LLM-DHRP (DM Universe) ===
from src.training.trainer import train_dhrp, train_llm_dhrp, train_llm_dhrp_warmstart

# Out-of-sample split
OOS_START = '2020-07-01'
TRAIN_END = '2020-06-30'
print(f'Train period: {START} to {TRAIN_END}')
print(f'OOS test period: {OOS_START} to {END}')

print('\n=== DEVELOPED MARKETS ===')

# DHRP -- the headline contribution
print('\n--- Training DHRP (price-only) ---')
dhrp_dm = train_dhrp(DM_prices, device=device, is_em=False, volume=DM_vol, train_end=TRAIN_END)

# LLM-DHRP -- warm-start from pretrained DHRP
# Fixes applied: gate_bias=-1.0, text_lr_scale=0.5, modality_dropout=0.1 (universe_config)
# Warm-start: freeze price pathway first, then fine-tune all
llm_dhrp_dm = None
if text_dm is not None:
    print('\n--- Training LLM-DHRP DM (warm-start) ---')
    llm_dhrp_dm = train_llm_dhrp_warmstart(
        DM_prices,
        pretrained_dhrp=dhrp_dm,
        text_features={'finbert': text_dm},
        macro_features=dm_macro.values if dm_macro is not None else None,
        device=device, is_em=False,
        use_text=True, use_macro=dm_macro is not None,
        fusion_type='cross_attention',
        epochs=60, lr=2e-4,
        volume=DM_vol,
        train_end=TRAIN_END,
        universe='DM',
    )

    # Run diagnostics
    from scripts.diagnose_llm_dhrp import run_full_diagnostics
    from src.data.feature_engineering import build_dataset
    from src.data.llm_features import aggregate_text_per_timestep
    import numpy as np
    X_diag, S_diag, R_diag, _ = build_dataset(DM_prices, train_end=TRAIN_END, volume=DM_vol)
    n_text = min(text_dm.shape[0], X_diag.shape[0])
    # Match training-time aggregation (norm_mean_max_concat -> 1536 dim)
    text_embs_diag = aggregate_text_per_timestep(
        text_dm[:n_text], method="norm_mean_max_concat"
    )
    if n_text < X_diag.shape[0]:
        text_embs_diag = np.vstack([text_embs_diag,
            np.zeros((X_diag.shape[0] - n_text, text_embs_diag.shape[1]), dtype=np.float32)])
    diag_dm = run_full_diagnostics(
        llm_dhrp_dm, X_diag, S_diag, text_embs_diag,
        device=device, label='DM',
    )

# Save models
torch.save(dhrp_dm.state_dict(), 'results/models/dhrp_dm.pt')
if llm_dhrp_dm is not None:
    torch.save(llm_dhrp_dm.state_dict(), 'results/models/llm_dhrp_dm.pt')
print('\nDM models saved.')

In [9]:
# === CELL 9: TRAIN EM + COMMODITIES ===

print('=== EMERGING MARKETS ===')
print('\n--- Training DHRP (EM, price-only) ---')
dhrp_em = train_dhrp(EM_prices, device=device, is_em=True, volume=EM_vol, train_end=TRAIN_END)

llm_dhrp_em = None
if text_em is not None:
    print('\n--- Training LLM-DHRP EM (warm-start) ---')
    llm_dhrp_em = train_llm_dhrp_warmstart(
        EM_prices,
        pretrained_dhrp=dhrp_em,
        text_features={'finbert': text_em},
        macro_features=dm_macro.values if dm_macro is not None else None,
        device=device, is_em=True,
        use_text=True, use_macro=dm_macro is not None,
        epochs=50, lr=1.5e-4, volume=EM_vol,
        train_end=TRAIN_END,
        universe='EM',
    )

print('\n=== COMMODITIES ===')
print('\n--- Training DHRP (Commodities, price-only) ---')
dhrp_cmd = train_dhrp(CMD_prices, device=device, is_em=False, volume=CMD_vol, train_end=TRAIN_END)

llm_dhrp_cmd = None
if text_cmd is not None:
    print('\n--- Training LLM-DHRP Commodities (warm-start) ---')
    llm_dhrp_cmd = train_llm_dhrp_warmstart(
        CMD_prices,
        pretrained_dhrp=dhrp_cmd,
        text_features={'finbert': text_cmd},
        macro_features=dm_macro.values if dm_macro is not None else None,
        device=device, is_em=False,
        use_text=True, use_macro=dm_macro is not None,
        epochs=60, lr=2e-4, volume=CMD_vol,
        train_end=TRAIN_END,
        universe='Commodities',
    )

# Save all models
torch.save(dhrp_em.state_dict(), 'results/models/dhrp_em.pt')
torch.save(dhrp_cmd.state_dict(), 'results/models/dhrp_cmd.pt')
if llm_dhrp_em: torch.save(llm_dhrp_em.state_dict(), 'results/models/llm_dhrp_em.pt')
if llm_dhrp_cmd: torch.save(llm_dhrp_cmd.state_dict(), 'results/models/llm_dhrp_cmd.pt')
print('\nAll models saved.')

=== EMERGING MARKETS ===

--- Training DHRP (EM, price-only) ---
  [EM] 362 samples, 10 assets, fdim=64
  [EM] Epoch 1/50, loss=-0.325945
  [EM] Epoch 10/50, loss=-0.317346
  [EM] Epoch 20/50, loss=-0.336327
  [EM] Epoch 30/50, loss=-0.313604
  [EM] Epoch 40/50, loss=-0.354821
  [EM] Epoch 50/50, loss=-0.325536

--- Training LLM-DHRP EM (warm-start) ---
  [EM] 362 samples (warm-start LLM-DHRP)
  [EM] Text features: (362, 768)
  [EM] Transferred 8 params from pretrained DHRP
  [EM] Phase 1: training text pathway only (30 epochs)
  [EM] Phase 1 Epoch 1/30, loss=-0.325411
  [EM] Phase 1 Epoch 10/30, loss=-0.327721
  [EM] Phase 1 Epoch 20/30, loss=-0.292800
  [EM] Phase 1 Epoch 30/30, loss=-0.334855
  [EM] Phase 2: fine-tuning all params (20 epochs)
  [EM] Phase 2 Epoch 1/20, loss=-0.334292
  [EM] Phase 2 Epoch 10/20, loss=-0.323465
  [EM] Phase 2 Epoch 20/20, loss=-0.325321

=== COMMODITIES ===

--- Training DHRP (Commodities, price-only) ---
  [DM] 362 samples, 10 assets, fdim=64
  [DM] 

In [10]:
# === CELL 10: ROLLING BACKTEST (all universes, OOS only) ===
from src.evaluation.backtest import rolling_backtest
import pandas as pd

METHODS = ['EW', 'MINVAR', 'MV', 'HRP', 'RP', 'MAXDIV', 'DHRP']
if llm_dhrp_dm is not None:
    METHODS.append('LLM_DHRP')

# Weight EMA smoothing for neural methods (reduces turnover)
WEIGHT_EMA = 0.3  # blend 30% previous + 70% new weights

print(f'Running OOS backtests (from {OOS_START})...\n')

dm_res = rolling_backtest(
    DM_prices, is_em=False, dhrp_model=dhrp_dm,
    llm_dhrp_model=llm_dhrp_dm,
    text_features={'finbert': text_dm_pit} if text_dm_pit is not None else None,
    macro_features=dm_macro, methods=METHODS, volume=DM_vol,
    oos_start=OOS_START, universe='DM',
    weight_ema=WEIGHT_EMA,
)
print(f'DM: {len(dm_res)} observations')

em_res = rolling_backtest(
    EM_prices, is_em=True, dhrp_model=dhrp_em,
    llm_dhrp_model=llm_dhrp_em,
    text_features={'finbert': text_em_pit} if text_em_pit is not None else None,
    macro_features=dm_macro, methods=METHODS, volume=EM_vol,
    oos_start=OOS_START, universe='EM',
    weight_ema=WEIGHT_EMA,
)
print(f'EM: {len(em_res)} observations')

cmd_res = rolling_backtest(
    CMD_prices, is_em=False, dhrp_model=dhrp_cmd,
    llm_dhrp_model=llm_dhrp_cmd,
    text_features={'finbert': text_cmd_pit} if text_cmd_pit is not None else None,
    macro_features=dm_macro, methods=METHODS, volume=CMD_vol,
    oos_start=OOS_START, universe='Commodities',
    weight_ema=WEIGHT_EMA,
)
print(f'Commodities: {len(cmd_res)} observations')

for _lbl, _r in [('DM', dm_res), ('EM', em_res), ('CMD', cmd_res)]:
    if not _r.empty:
        _min = pd.to_datetime(_r['date']).min()
        _max = pd.to_datetime(_r['date']).max()
        assert _min >= pd.Timestamp(OOS_START), f'{_lbl}: OOS leak -- {_min} < {OOS_START}'
        print(f'  {_lbl} OOS span: {_min.date()} -> {_max.date()} ({_r["date"].nunique()} days)')

Running OOS backtests (from 2020-07-01)...

DM: 11496 observations
EM: 11496 observations
Commodities: 11624 observations
  DM OOS span: 2020-07-29 -> 2026-04-17 (1437 days)
  EM OOS span: 2020-07-29 -> 2026-04-17 (1437 days)
  CMD OOS span: 2020-07-07 -> 2026-04-17 (1453 days)


In [11]:
# === CELL 11: RESULTS & STATISTICAL TESTS ===
from src.evaluation.statistics import compute_stats, sharpe_difference_test, diebold_mariano_test, subperiod_analysis
from src.evaluation.factor_analysis import factor_analysis, load_aqr_commodity_factors, commodity_factor_analysis
import pandas as pd

os.makedirs('results/full', exist_ok=True)

for label, res, prices, is_em in [
    ('DM', dm_res, DM_prices, False),
    ('EM', em_res, EM_prices, True),
    ('Commodities', cmd_res, CMD_prices, False),
]:
    sep = "=" * 60
    print()
    print(sep)
    print("  " + label + " RESULTS (" + str(prices.shape[1]) + " assets)")
    print(sep)
    
    stats = compute_stats(res, oos_start=OOS_START)
    try:
        factors = factor_analysis(res, FF)
        table = stats.merge(factors, on='Method', how='left').round(3)
    except Exception:
        table = stats.round(3)
    print(table.to_string(index=False))
    
    # Commodity-specific factor analysis using AQR Value & Momentum
    if label == 'Commodities':
        try:
            aqr = load_aqr_commodity_factors(
                prices.index[0].strftime('%Y-%m-%d'),
                prices.index[-1].strftime('%Y-%m-%d'),
            )
            cm_factors = commodity_factor_analysis(res, aqr)
            print("\n--- Commodity Factor Analysis (AQR Value & Momentum) ---")
            print(cm_factors.to_string(index=False))
            cm_factors.to_csv('results/full/' + label + '_aqr_factors.csv', index=False)
        except Exception as e:
            print('  AQR commodity factors failed: ' + str(e))
    
    # Statistical tests vs HRP
    print("\n--- Statistical Tests (vs HRP) ---")
    for m in sorted(res['method'].unique()):
        if m == 'HRP':
            continue
        try:
            diff = sharpe_difference_test(res, m, 'HRP')
            dm_test = diebold_mariano_test(res, m, 'HRP')
            bp = diff["bootstrap_p"]
            sig = " ***" if bp < 0.01 else " **" if bp < 0.05 else " *" if bp < 0.10 else ""
            msg = "  {:12s} vs HRP: Sharpe diff={:+.3f} [{:.3f}, {:.3f}] p={:.3f}{} | DM stat={:+.3f} p={:.3f}".format(
                m, diff["sharpe_diff"], diff["bootstrap_ci_lo"], diff["bootstrap_ci_hi"],
                bp, sig, dm_test["DM_stat"], dm_test["p_value"])
            print(msg)
        except Exception as e:
            print('  ' + m + ' vs HRP: failed (' + str(e) + ')')
    
    # Save results
    table.to_csv('results/' + label + '_results.csv', index=False)
    table.to_csv('results/full/' + label + '_stats.csv', index=False)
    sub = subperiod_analysis(res, train_end=TRAIN_END)
    if not sub.empty:
        print('\n--- Sub-period Sharpe Ratios ---')
        print(sub.pivot(index='Method', columns='Period', values='Sharpe').round(3).to_string())
    res.to_csv('results/full/' + label + '_backtest.csv', index=False)


  DM RESULTS (10 assets)
  Method  Sharpe  Sortino  Calmar  MaxDD  CVaR_5  VaR_5  Omega   CER  Ann_Return  Ann_Vol   HAC_t  CI_lo  CI_hi  Alpha_ann  Alpha_t  Alpha_p  R2_adj  Beta_Mkt-RF  t_Mkt-RF  Beta_SMB  t_SMB  Beta_HML   t_HML
    DHRP   0.133    0.188   0.054 -0.239  -0.014 -0.010  1.077 0.031       0.013    0.098   5.217 -0.602  0.947     -0.010   -0.516    0.606   0.729        0.442    21.524     0.094  5.234    -0.035  -1.948
      EW   0.248    0.362   0.109 -0.247  -0.016 -0.011  1.094 0.042       0.027    0.109   9.513 -0.492  1.049     -0.014   -0.737    0.461   0.849        0.544    52.507     0.111  8.503     0.010   0.914
     HRP   0.207    0.300   0.088 -0.205  -0.012 -0.009  1.100 0.039       0.018    0.087   7.955 -0.538  1.017     -0.007   -0.409    0.683   0.796        0.425    45.220     0.052  4.213    -0.007  -0.711
LLM_DHRP   0.043    0.061   0.019 -0.199  -0.013 -0.009  1.067 0.024       0.004    0.088   1.695 -0.716  0.897     -0.004   -0.199    0.842   0.6

## Publication Benchmarks: Deep Baselines, Ablations, Robustness, Statistical Tests

The following cells run the full benchmark suite required for top AI/ML venues (NeurIPS, ICML, ICLR).

In [12]:
# === BENCH 1: TRAIN DEEP BASELINES (MLP, Transformer, PPO) ===
from src.models.deep_baselines import train_ppo_agent, train_transformer_policy
from src.models.deep_baselines import MLPWithCovPolicy
from src.models.loss_functions import dhrp_loss
from src.data.feature_engineering import build_dataset, DEFAULT_FDIM
import time

print('=== Training Deep Learning Baselines (DM, in-sample only) ===\n')

# MLP baseline (train on in-sample only)
print('--- MLP ---')
t0 = time.perf_counter()
X, S, R, H = build_dataset(DM_prices, volume=DM_vol, train_end=TRAIN_END)
fdim_actual, n_assets = X.shape[1], DM_prices.shape[1]
mlp_dm = MLPWithCovPolicy(fdim_actual, n_assets).to(device)
mlp_opt = torch.optim.AdamW(mlp_dm.parameters(), lr=3e-4, weight_decay=3e-4)
Xt = torch.from_numpy(X).to(device)
St = torch.from_numpy(S).to(device)
Rt = torch.from_numpy(R).to(device)
best_loss_mlp, best_st_mlp = float('inf'), None
for ep in range(40):
    perm = torch.randperm(X.shape[0])
    ep_loss, nb = 0.0, 0
    for s in range(0, X.shape[0], 32):
        e = min(s + 32, X.shape[0])
        mlp_opt.zero_grad()
        loss = dhrp_loss(mlp_dm, Xt[perm[s:e]], St[perm[s:e]], Rt[perm[s:e]],
                         H[perm[s:e].cpu().numpy()], is_em=False, lam_hrp=0.1)
        if not torch.isnan(loss) and loss.requires_grad:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mlp_dm.parameters(), 1.0)
            mlp_opt.step()
            ep_loss += loss.item(); nb += 1
    if nb > 0 and ep_loss / nb < best_loss_mlp:
        best_loss_mlp = ep_loss / nb
        best_st_mlp = {k: v.cpu().clone() for k, v in mlp_dm.state_dict().items()}
    if (ep + 1) % 10 == 0:
        print(f'  Epoch {ep+1}/40, loss={ep_loss/max(nb,1):.6f}')
if best_st_mlp:
    mlp_dm.load_state_dict({k: v.to(device) for k, v in best_st_mlp.items()})
print(f'  MLP trained in {time.perf_counter()-t0:.1f}s')

# Transformer baseline
print('\n--- Transformer ---')
transformer_dm = train_transformer_policy(
    DM_prices, device=device, volume=DM_vol, train_end=TRAIN_END,
)

# PPO baseline
print('\n--- PPO ---')
ppo_dm = train_ppo_agent(
    DM_prices, device=device, volume=DM_vol, train_end=TRAIN_END,
)

# Save deep baselines
torch.save(mlp_dm.state_dict(), 'results/models/mlp_dm.pt')
torch.save(transformer_dm.state_dict(), 'results/models/transformer_dm.pt')
torch.save(ppo_dm.state_dict(), 'results/models/ppo_dm.pt')
print('\nDeep baselines saved.')


# --- DFL (Decision-Focused Learning baseline) ---
print('\n--- DFL ---')
from src.models.deep_baselines import train_dfl_baseline
dfl_dm = train_dfl_baseline(
    DM_prices, device=device, volume=DM_vol, train_end=TRAIN_END,
)
torch.save(dfl_dm.state_dict(), 'results/models/dfl_dm.pt')
print('DFL trained and saved.')

=== Training Deep Learning Baselines (DM, in-sample only) ===

--- MLP ---
  Epoch 10/40, loss=-0.177921
  Epoch 20/40, loss=-0.235221
  Epoch 30/40, loss=-0.353585
  Epoch 40/40, loss=-0.396755
  MLP trained in 32.9s

--- Transformer ---
  [Transformer] Epoch 1/40, loss=-0.065834
  [Transformer] Epoch 10/40, loss=-0.214439
  [Transformer] Epoch 20/40, loss=-0.286096
  [Transformer] Epoch 30/40, loss=-0.382483
  [Transformer] Epoch 40/40, loss=-0.439376

--- PPO ---
  [PPO] Epoch 1/40, avg_reward=0.001258
  [PPO] Epoch 10/40, avg_reward=0.001362
  [PPO] Epoch 20/40, avg_reward=0.001001
  [PPO] Epoch 30/40, avg_reward=0.001438
  [PPO] Epoch 40/40, avg_reward=0.001275

Deep baselines saved.

--- DFL ---
  [DFL] 362 samples, 10 assets, fdim=64
  [DFL] Epoch 1/40, loss=-0.128226
  [DFL] Epoch 10/40, loss=-0.283139
  [DFL] Epoch 20/40, loss=-0.364029
  [DFL] Epoch 30/40, loss=-0.472432
  [DFL] Epoch 40/40, loss=-0.500959
DFL trained and saved.


In [13]:
# === BENCH 2: BACKTEST WITH ALL METHODS (including deep baselines, OOS) ===
from src.evaluation.backtest import rolling_backtest

ALL_METHODS = ['EW', 'MINVAR', 'MV', 'HRP', 'RP', 'MAXDIV', 'DHRP', 'MLP', 'Transformer', 'PPO', 'DFL']
if llm_dhrp_dm is not None:
    ALL_METHODS.append('LLM_DHRP')

print(f'Running full OOS backtest with all methods (DM, from {OOS_START})...')
dm_full, dm_weights = rolling_backtest(
    DM_prices, is_em=False,
    dhrp_model=dhrp_dm, llm_dhrp_model=llm_dhrp_dm,
    mlp_model=mlp_dm, transformer_model=transformer_dm, ppo_model=ppo_dm,
    dfl_model=dfl_dm,
    text_features={'finbert': text_dm_pit} if text_dm_pit is not None else None,
    macro_features=dm_macro,
    methods=ALL_METHODS, return_weights=True, volume=DM_vol,
    oos_start=OOS_START, weight_ema=0.3,
)
print(f'DM full: {len(dm_full)} obs, {dm_full["method"].nunique()} methods')

from src.evaluation.statistics import (
    compute_stats, full_statistical_battery, subperiod_analysis,
    compute_turnover, cost_sensitivity_analysis, benchmark_efficiency,
    spa_test, model_confidence_set,
)

print('\n=== COMPREHENSIVE METRICS (DM, OOS) ===')
dm_full_stats = compute_stats(dm_full, gamma=2.5, oos_start=OOS_START)
print(dm_full_stats.round(3).to_string(index=False))
dm_full_stats.to_csv('results/DM_full_stats.csv', index=False)

# SPA test and Model Confidence Set
print('\n--- Superior Predictive Ability (SPA) Test ---')
spa = spa_test(dm_full, benchmark='EW', n_boot=2000)
print(f'  SPA p-value: {spa["spa_pvalue"]:.4f}')
print(f'  Best method: {spa["best_method"]} (excess Sharpe: {spa["best_excess_sharpe"]:.3f})')

print('\n--- Model Confidence Set (alpha=0.05) ---')
mcs = model_confidence_set(dm_full, alpha=0.05, n_boot=2000)
print(f'  MCS members: {mcs}')

Running full OOS backtest with all methods (DM, from 2020-07-01)...
DM full: 17244 obs, 12 methods

=== COMPREHENSIVE METRICS (DM, OOS) ===
     Method  Sharpe  Sortino  Calmar  MaxDD  CVaR_5  VaR_5  Omega   CER  Ann_Return  Ann_Vol   HAC_t  CI_lo  CI_hi
        DFL   0.371    0.520   0.191 -0.209  -0.016 -0.011  1.120 0.055       0.040    0.107  15.282 -0.264  1.259
       DHRP   0.133    0.188   0.054 -0.239  -0.014 -0.010  1.077 0.031       0.013    0.098   5.217 -0.602  0.947
         EW   0.248    0.362   0.109 -0.247  -0.016 -0.011  1.094 0.042       0.027    0.109   9.513 -0.492  1.049
        HRP   0.207    0.300   0.088 -0.205  -0.012 -0.009  1.100 0.039       0.018    0.087   7.955 -0.538  1.017
   LLM_DHRP   0.043    0.061   0.019 -0.199  -0.013 -0.009  1.067 0.024       0.004    0.088   1.695 -0.716  0.897
     MAXDIV  -0.407   -0.480  -0.285 -0.061  -0.007 -0.004  1.055 0.010      -0.017    0.043 -16.396 -1.131  0.351
     MINVAR  -0.608   -0.701  -0.455 -0.050  -0.005 -0.

In [14]:
# === BENCH 3: PAIRWISE STATISTICAL TESTS (Holm-Bonferroni) ===
from src.evaluation.statistics import full_statistical_battery, pairwise_sharpe_tests, pairwise_dm_tests

# Run pairwise tests for ALL THREE universes
for label, res_data, ref_method in [
    ('DM', dm_full, 'DHRP'),
    ('EM', em_res, 'DHRP'),
    ('Commodities', cmd_res, 'DHRP'),
]:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  PAIRWISE TESTS: {label} (ref={ref_method})')
    print(f'{sep}')

    # Bootstrap Sharpe ratio difference tests
    print(f'\n--- Bootstrap Sharpe Tests (Holm-Bonferroni) ---')
    sharpe_tests = pairwise_sharpe_tests(res_data, ref=ref_method, n_boot=2000)
    if not sharpe_tests.empty:
        cols = ['method_b', 'sharpe_diff', 'bootstrap_ci_lo', 'bootstrap_ci_hi',
                'bootstrap_p', 'adjusted_p', 'significant_005']
        print(sharpe_tests[cols].round(3).to_string(index=False))
        sharpe_tests.to_csv(f'results/{label}_sharpe_tests.csv', index=False)

    # Diebold-Mariano tests (squared loss)
    print(f'\n--- Diebold-Mariano Tests (squared loss) ---')
    dm_tests_sq = pairwise_dm_tests(res_data, ref=ref_method, loss_fn='squared')
    if not dm_tests_sq.empty:
        print(dm_tests_sq[['method_b', 'DM_stat', 'p_value', 'adjusted_p', 'significant_005']].round(3).to_string(index=False))
        dm_tests_sq.to_csv(f'results/{label}_dm_tests_squared.csv', index=False)

    # Diebold-Mariano tests (negative return)
    print(f'\n--- Diebold-Mariano Tests (negative return) ---')
    dm_tests_neg = pairwise_dm_tests(res_data, ref=ref_method, loss_fn='negative')
    if not dm_tests_neg.empty:
        print(dm_tests_neg[['method_b', 'DM_stat', 'p_value', 'adjusted_p', 'significant_005']].round(3).to_string(index=False))
        dm_tests_neg.to_csv(f'results/{label}_dm_tests_negative.csv', index=False)

print('\nPairwise tests complete for all universes.')
# SPA and MCS for all universes
from src.evaluation.statistics import spa_test, model_confidence_set
print('\n--- SPA & MCS Summary ---')
for label, res_data in [('DM', dm_full), ('EM', em_res), ('Commodities', cmd_res)]:
    spa = spa_test(res_data, benchmark='EW', n_boot=2000)
    mcs = model_confidence_set(res_data, alpha=0.05, n_boot=2000)
    print(f'  {label}: SPA p={spa["spa_pvalue"]:.4f}, best={spa["best_method"]}, MCS={mcs}')



  PAIRWISE TESTS: DM (ref=DHRP)

--- Bootstrap Sharpe Tests (Holm-Bonferroni) ---
   method_b  sharpe_diff  bootstrap_ci_lo  bootstrap_ci_hi  bootstrap_p  adjusted_p  significant_005
        DFL       -0.238           -0.629            0.187        0.243         1.0            False
         EW       -0.115           -0.403            0.196        0.462         1.0            False
        HRP       -0.075           -0.394            0.274        0.656         1.0            False
   LLM_DHRP        0.090           -0.277            0.459        0.637         1.0            False
     MAXDIV        0.539           -0.082            1.143        0.084         1.0            False
     MINVAR        0.741            0.024            1.399        0.035         0.8            False
        MLP       -0.195           -0.583            0.183        0.318         1.0            False
         MV       -0.530           -1.188            0.166        0.127         1.0            False
        

In [15]:
# === BENCH 4: ABLATION STUDIES ===
from src.models.dhrp_layer import DHRPLayer
from src.models.loss_functions import dhrp_loss
from src.training.trainer import train_dhrp
from src.evaluation.backtest import rolling_backtest
from src.evaluation.statistics import compute_stats

print('=== ABLATION STUDIES (DM) ===\n')
ablation_rows = []

X, S, R, H = build_dataset(DM_prices, volume=DM_vol, train_end=TRAIN_END)
n_assets = DM_prices.shape[1]
Xt = torch.from_numpy(X).to(device)
St = torch.from_numpy(S).to(device)
Rt = torch.from_numpy(R).to(device)


def train_ablation(model, lam_hrp_fn, label, config):
    """Train a DHRP model variant and backtest it."""
    opt = torch.optim.AdamW(model.parameters(), lr=4.5e-4, weight_decay=3e-4)
    best_l, best_s = float('inf'), None
    for ep in range(40):
        perm = torch.randperm(X.shape[0])
        el, nb_ = 0.0, 0
        lam = lam_hrp_fn(ep)
        for s in range(0, X.shape[0], 32):
            e = min(s + 32, X.shape[0])
            opt.zero_grad()
            loss = dhrp_loss(model, Xt[perm[s:e]], St[perm[s:e]], Rt[perm[s:e]],
                             H[perm[s:e].cpu().numpy()], is_em=False, lam_hrp=lam)
            if not torch.isnan(loss) and loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                el += loss.item(); nb_ += 1
        if nb_ > 0 and el / nb_ < best_l:
            best_l = el / nb_
            best_s = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_s:
        model.load_state_dict({k: v.to(device) for k, v in best_s.items()})
    res = rolling_backtest(DM_prices, is_em=False, dhrp_model=model, methods=['DHRP'], volume=DM_vol, oos_start=OOS_START)
    stats = compute_stats(res)
    if not stats.empty:
        row = stats.iloc[0].to_dict()
        row['Ablation'] = label
        row['Config'] = config
        ablation_rows.append(row)
        print(f'  {label} [{config}]: Sharpe={row["Sharpe"]:.3f}')
    return model


# --- A1: Tree depth ablation ---
print('--- A1: Tree Depth ---')
for depth in [2, 3, 4]:
    model = DHRPLayer(n_assets, X.shape[1], hidden_dim=64, depth=depth, is_em=False).to(device)
    train_ablation(model, lambda ep: 0.3 - 0.2 * (ep / 40), 'Tree depth', f'depth={depth}')

# --- A2: Loss component ablation ---
print('\n--- A2: Loss Components ---')

# Full model (baseline)
model_full = DHRPLayer(n_assets, X.shape[1], hidden_dim=64, depth=3, is_em=False).to(device)
train_ablation(model_full, lambda ep: 0.3 - 0.2 * (ep / 40), 'Loss component', 'Full (baseline)')

# No HRP regularization
model_nreg = DHRPLayer(n_assets, X.shape[1], hidden_dim=64, depth=3, is_em=False).to(device)
train_ablation(model_nreg, lambda ep: 0.0, 'Loss component', 'No HRP reg')

# No Sharpe term
print('  Training: No Sharpe term...')
model_nosharpe = DHRPLayer(n_assets, X.shape[1], hidden_dim=64, depth=3, is_em=False).to(device)
opt_ns = torch.optim.AdamW(model_nosharpe.parameters(), lr=4.5e-4, weight_decay=3e-4)
best_l, best_s = float('inf'), None
for ep in range(40):
    perm = torch.randperm(X.shape[0])
    el, nb_ = 0.0, 0
    lam = 0.3 - 0.2 * (ep / 40)
    for s in range(0, X.shape[0], 32):
        e = min(s + 32, X.shape[0])
        opt_ns.zero_grad()
        port_r, wts = [], []
        for t in range(Rt[perm[s:e]].shape[0]):
            w = model_nosharpe(Xt[perm[s:e]][t], St[perm[s:e]][t])
            wts.append(w); port_r.append((w * Rt[perm[s:e]][t]).sum())
        port_r = torch.stack(port_r); wts = torch.stack(wts)
        crra = ((torch.clamp(1 + port_r, min=0.1) ** (1 - 2.5) - 1) / (1 - 2.5)).mean()
        hrp_target = torch.from_numpy(H[perm[s:e].cpu().numpy()]).to(device).float()
        hrp_reg = ((wts - hrp_target) ** 2).mean() * lam * 0.2
        risk = torch.stack([wts[t] @ St[perm[s:e]][t] @ wts[t] for t in range(port_r.shape[0])]).mean() * 0.001
        hhi = (wts ** 2).sum(1).mean() * 0.1
        loss = -crra + hrp_reg + risk + hhi  # No sharpe
        if not torch.isnan(loss) and loss.requires_grad:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model_nosharpe.parameters(), 1.0)
            opt_ns.step(); el += loss.item(); nb_ += 1
    if nb_ > 0 and el / nb_ < best_l:
        best_l = el / nb_; best_s = {k: v.cpu().clone() for k, v in model_nosharpe.state_dict().items()}
if best_s:
    model_nosharpe.load_state_dict({k: v.to(device) for k, v in best_s.items()})
res = rolling_backtest(DM_prices, is_em=False, dhrp_model=model_nosharpe, methods=['DHRP'], volume=DM_vol, oos_start=OOS_START)
stats = compute_stats(res)
if not stats.empty:
    row = stats.iloc[0].to_dict(); row['Ablation'] = 'Loss component'; row['Config'] = 'No Sharpe'
    ablation_rows.append(row); print(f'  Loss component [No Sharpe]: Sharpe={row["Sharpe"]:.3f}')

# No CRRA term
print('  Training: No CRRA term...')
model_nocrra = DHRPLayer(n_assets, X.shape[1], hidden_dim=64, depth=3, is_em=False).to(device)
opt_nc = torch.optim.AdamW(model_nocrra.parameters(), lr=4.5e-4, weight_decay=3e-4)
best_l, best_s = float('inf'), None
for ep in range(40):
    perm = torch.randperm(X.shape[0])
    el, nb_ = 0.0, 0
    lam = 0.3 - 0.2 * (ep / 40)
    for s in range(0, X.shape[0], 32):
        e = min(s + 32, X.shape[0])
        opt_nc.zero_grad()
        port_r, wts = [], []
        for t in range(Rt[perm[s:e]].shape[0]):
            w = model_nocrra(Xt[perm[s:e]][t], St[perm[s:e]][t])
            wts.append(w); port_r.append((w * Rt[perm[s:e]][t]).sum())
        port_r = torch.stack(port_r); wts = torch.stack(wts)
        sharpe = port_r.mean() / (port_r.std() + 1e-6)
        hrp_target = torch.from_numpy(H[perm[s:e].cpu().numpy()]).to(device).float()
        hrp_reg = ((wts - hrp_target) ** 2).mean() * lam * 0.2
        risk = torch.stack([wts[t] @ St[perm[s:e]][t] @ wts[t] for t in range(port_r.shape[0])]).mean() * 0.001
        hhi = (wts ** 2).sum(1).mean() * 0.1
        loss = -sharpe + hrp_reg + risk + hhi  # No CRRA
        if not torch.isnan(loss) and loss.requires_grad:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model_nocrra.parameters(), 1.0)
            opt_nc.step(); el += loss.item(); nb_ += 1
    if nb_ > 0 and el / nb_ < best_l:
        best_l = el / nb_; best_s = {k: v.cpu().clone() for k, v in model_nocrra.state_dict().items()}
if best_s:
    model_nocrra.load_state_dict({k: v.to(device) for k, v in best_s.items()})
res = rolling_backtest(DM_prices, is_em=False, dhrp_model=model_nocrra, methods=['DHRP'], volume=DM_vol, oos_start=OOS_START)
stats = compute_stats(res)
if not stats.empty:
    row = stats.iloc[0].to_dict(); row['Ablation'] = 'Loss component'; row['Config'] = 'No CRRA'
    ablation_rows.append(row); print(f'  Loss component [No CRRA]: Sharpe={row["Sharpe"]:.3f}')

# --- A3: LLM-DHRP Text Fusion Ablation ---
print('\n--- A3: Text Fusion Ablation (LLM-DHRP) ---')
if llm_dhrp_dm is not None and text_dm is not None:
    # Full LLM-DHRP (reference)
    res = rolling_backtest(
        DM_prices, is_em=False, llm_dhrp_model=llm_dhrp_dm,
        text_features={'finbert': text_dm}, methods=['LLM_DHRP'], volume=DM_vol, oos_start=OOS_START,
    )
    stats = compute_stats(res)
    if not stats.empty:
        row = stats.iloc[0].to_dict()
        row['Ablation'] = 'Text fusion'; row['Config'] = 'Full LLM-DHRP'
        ablation_rows.append(row)
        print(f'  Text fusion [Full LLM-DHRP]: Sharpe={row["Sharpe"]:.3f}')

    # DHRP without any text (reference)
    res = rolling_backtest(
        DM_prices, is_em=False, dhrp_model=dhrp_dm,
        methods=['DHRP'], volume=DM_vol, oos_start=OOS_START,
    )
    stats = compute_stats(res)
    if not stats.empty:
        row = stats.iloc[0].to_dict()
        row['Ablation'] = 'Text fusion'; row['Config'] = 'No text (DHRP)'
        ablation_rows.append(row)
        print(f'  Text fusion [No text (DHRP)]: Sharpe={row["Sharpe"]:.3f}')
else:
    print('  Skipping text fusion ablation (LLM-DHRP or text features not available)')

# Compile ablation table
ablation_df = pd.DataFrame(ablation_rows)
print('\n=== ABLATION SUMMARY ===')
if not ablation_df.empty:
    cols = [c for c in ['Ablation', 'Config', 'Sharpe', 'Sortino', 'MaxDD', 'Calmar'] if c in ablation_df.columns]
    print(ablation_df[cols].round(3).to_string(index=False))
ablation_df.to_csv('results/DM_ablations.csv', index=False)
print('Ablation results saved.')

=== ABLATION STUDIES (DM) ===

--- A1: Tree Depth ---
  Tree depth [depth=2]: Sharpe=0.268
  Tree depth [depth=3]: Sharpe=0.163
  Tree depth [depth=4]: Sharpe=-0.044

--- A2: Loss Components ---
  Loss component [Full (baseline)]: Sharpe=0.162
  Loss component [No HRP reg]: Sharpe=0.162
  Training: No Sharpe term...
  Loss component [No Sharpe]: Sharpe=0.308
  Training: No CRRA term...
  Loss component [No CRRA]: Sharpe=0.250

--- A3: Text Fusion Ablation (LLM-DHRP) ---
  Text fusion [Full LLM-DHRP]: Sharpe=0.041
  Text fusion [No text (DHRP)]: Sharpe=0.075

=== ABLATION SUMMARY ===
      Ablation          Config  Sharpe  Sortino  MaxDD  Calmar
    Tree depth         depth=2   0.268    0.389 -0.208   0.137
    Tree depth         depth=3   0.163    0.232 -0.221   0.079
    Tree depth         depth=4  -0.044   -0.057 -0.189  -0.026
Loss component Full (baseline)   0.162    0.231 -0.204   0.085
Loss component      No HRP reg   0.162    0.225 -0.233   0.076
Loss component       No Sharpe  

In [16]:
# === BENCH 5: MULTI-SEED ROBUSTNESS ===
from src.training.trainer import train_dhrp_multiseed
from src.evaluation.backtest import multiseed_backtest

print('=== MULTI-SEED ROBUSTNESS (DM, 5 seeds) ===\n')

dhrp_models = train_dhrp_multiseed(
    DM_prices, seeds=[0, 1, 2, 3, 4], device=device, is_em=False,
    train_end=TRAIN_END,
)

models_by_seed = [{'dhrp': m} for m in dhrp_models]

seed_agg = multiseed_backtest(
    DM_prices, models_by_seed, is_em=False,
    methods=['EW', 'MINVAR', 'MV', 'HRP', 'RP', 'MAXDIV', 'DHRP'],
    oos_start=OOS_START,
)
print('\n=== MULTI-SEED AGGREGATED RESULTS ===')
print(seed_agg.round(3).to_string(index=False))
seed_agg.to_csv('results/DM_multiseed.csv', index=False)

# Collect per-seed stats for boxplot visualization
from src.evaluation.statistics import compute_stats as _cs
seed_stats_list = []
for i, m in enumerate(dhrp_models):
    res_i = rolling_backtest(
        DM_prices, is_em=False, dhrp_model=m,
        methods=['EW', 'MINVAR', 'MV', 'HRP', 'RP', 'MAXDIV', 'DHRP'],
        volume=DM_vol,
        oos_start=OOS_START,
    )
    s = _cs(res_i)
    s['seed'] = i
    seed_stats_list.append(s)
print(f'\nCollected per-seed stats for {len(seed_stats_list)} seeds')


=== MULTI-SEED ROBUSTNESS (DM, 5 seeds) ===


--- Seed 0 ---
  [DM] 362 samples, 10 assets, fdim=64
  [DM] Epoch 1/60, loss=-0.078781
  [DM] Epoch 10/60, loss=-0.204164
  [DM] Epoch 20/60, loss=-0.316852
  [DM] Epoch 30/60, loss=-0.427911
  [DM] Epoch 40/60, loss=-0.440917
  [DM] Epoch 50/60, loss=-0.472330
  [DM] Epoch 60/60, loss=-0.461451

--- Seed 1 ---
  [DM] 362 samples, 10 assets, fdim=64
  [DM] Epoch 1/60, loss=-0.121345
  [DM] Epoch 10/60, loss=-0.206524
  [DM] Epoch 20/60, loss=-0.353253
  [DM] Epoch 30/60, loss=-0.417660
  [DM] Epoch 40/60, loss=-0.479689
  [DM] Epoch 50/60, loss=-0.468146
  [DM] Epoch 60/60, loss=-0.461194

--- Seed 2 ---
  [DM] 362 samples, 10 assets, fdim=64
  [DM] Epoch 1/60, loss=-0.108835
  [DM] Epoch 10/60, loss=-0.256040
  [DM] Epoch 20/60, loss=-0.344969
  [DM] Epoch 30/60, loss=-0.404167
  [DM] Epoch 40/60, loss=-0.464242
  [DM] Epoch 50/60, loss=-0.478717
  [DM] Epoch 60/60, loss=-0.503442

--- Seed 3 ---
  [DM] 362 samples, 10 assets, fdim=64
  [

In [17]:
# === BENCH 6: TRANSACTION COST SENSITIVITY & TURNOVER ===
from src.evaluation.statistics import compute_turnover, cost_sensitivity_analysis, breakeven_cost

print('=== TRANSACTION COST ANALYSIS (DM) ===\n')

# Compute turnover from weight history (dm_weights from BENCH 2)
turnover_df = compute_turnover(dm_weights)
if not turnover_df.empty:
    print('--- Average Turnover per Method ---')
    for _, row in turnover_df.iterrows():
        ann = row["Avg_Turnover"] * (252 / max(row["N_Rebalances"], 1))
        print(f'  {row["Method"]:15s}: avg={row["Avg_Turnover"]:.4f}  '
              f'total={row["Total_Turnover"]:.2f}  rebalances={row["N_Rebalances"]:.0f}')
    turnover_df.to_csv('results/DM_turnover.csv', index=False)

# Cost sensitivity: Sharpe at 0, 5, 10, 20, 50 bps
print('\n--- Cost Sensitivity (Sharpe at various cost levels) ---')
cost_df = cost_sensitivity_analysis(dm_full, dm_weights, cost_levels=[0, 5, 10, 20, 50])
if not cost_df.empty:
    print(cost_df.round(3).to_string(index=False))
    cost_df.to_csv('results/DM_cost_sensitivity.csv', index=False)

# Breakeven cost for DHRP vs each baseline
print('\n--- Breakeven Transaction Cost (DHRP vs baselines) ---')
for baseline in ['EW', 'HRP', 'MINVAR', 'MV', 'RP', 'MAXDIV']:
    try:
        be = breakeven_cost(dm_full, dm_weights, method_a='DHRP', method_b=baseline)
        print(f'  DHRP vs {baseline:8s}: {be:.1f} bps')
    except Exception as e:
        print(f'  DHRP vs {baseline:8s}: N/A ({e})')

=== TRANSACTION COST ANALYSIS (DM) ===

--- Average Turnover per Method ---
  DFL            : avg=0.6393  total=43.47  rebalances=68
  DHRP           : avg=0.6170  total=41.95  rebalances=68
  EW             : avg=0.0000  total=0.00  rebalances=68
  HRP            : avg=0.0905  total=6.15  rebalances=68
  LLM_DHRP       : avg=0.0400  total=2.72  rebalances=68
  MAXDIV         : avg=0.0689  total=4.68  rebalances=68
  MINVAR         : avg=0.0585  total=3.98  rebalances=68
  MLP            : avg=0.4013  total=27.29  rebalances=68
  MV             : avg=0.3986  total=27.10  rebalances=68
  PPO            : avg=0.2439  total=16.59  rebalances=68
  RP             : avg=0.7318  total=49.76  rebalances=68
  Transformer    : avg=0.2186  total=14.86  rebalances=68

--- Cost Sensitivity (Sharpe at various cost levels) ---
     Method  cost_0bps  cost_5bps  cost_10bps  cost_20bps  cost_50bps
        DFL      0.371      0.335       0.300       0.229       0.016
       DHRP      0.133      0.095  

In [18]:
# === BENCH 7: REGIME-CONDITIONAL ANALYSIS ===
from src.evaluation.statistics import subperiod_analysis

print('=== REGIME-CONDITIONAL PERFORMANCE (DM) ===\n')

# 5 macro regimes defined in statistics.py:
# Pre-COVID bull, COVID crash, Recovery/stimulus, Rate hike cycle, Post-hike
regime_df = subperiod_analysis(dm_full)

if not regime_df.empty:
    print('--- Sharpe by Method x Regime ---')
    pivot_sharpe = regime_df.pivot(index='Method', columns='Period', values='Sharpe')
    print(pivot_sharpe.round(3).to_string())

    print('\n--- MaxDD by Method x Regime ---')
    pivot_dd = regime_df.pivot(index='Method', columns='Period', values='MaxDD')
    print(pivot_dd.round(3).to_string())

    print('\n--- Sortino by Method x Regime ---')
    pivot_sortino = regime_df.pivot(index='Method', columns='Period', values='Sortino')
    print(pivot_sortino.round(3).to_string())

    regime_df.to_csv('results/DM_regime_analysis.csv', index=False)
    print('\nRegime analysis saved.')
else:
    print('Regime analysis returned empty â€” check date coverage.')

# Also run for EM and Commodities
for label, res in [('EM', em_res), ('Commodities', cmd_res)]:
    print(f'\n--- {label} Regime Sharpe ---')
    rdf = subperiod_analysis(res)
    if not rdf.empty:
        pivot = rdf.pivot(index='Method', columns='Period', values='Sharpe')
        print(pivot.round(3).to_string())
        rdf.to_csv(f'results/{label}_regime_analysis.csv', index=False)

=== REGIME-CONDITIONAL PERFORMANCE (DM) ===

--- Sharpe by Method x Regime ---
Period       Post-Hike (2023-H2+)  Rate Hikes (2022 to 2023-H1)  Recovery (2020-H2 to 2021)
Method                                                                                     
DFL                         0.483                        -0.643                       1.159
DHRP                        0.148                        -0.794                       1.241
EW                          0.566                        -0.774                       1.265
HRP                         0.606                        -0.861                       1.109
LLM_DHRP                    0.340                        -0.823                       0.717
MAXDIV                     -0.098                        -0.989                      -0.524
MINVAR                     -0.259                        -0.954                      -1.203
MLP                         0.308                        -0.598                       1.213
M

In [19]:
# === BENCH 8: PUBLICATION FIGURES (NeurIPS-compatible, 300 DPI) ===
from src.visualization.plots import (
    plot_ablation_heatmap, plot_regime_bars, plot_cost_sensitivity,
    plot_seed_boxplots, plot_pairwise_dm_heatmap, plot_cumulative, plot_sharpe_bars,
)
from src.evaluation.statistics import benchmark_efficiency
import matplotlib.pyplot as plt

FIG_DIR = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

print('=== GENERATING PUBLICATION FIGURES ===\n')

# Fig 1: Cumulative returns (all 3 universes, all methods)
print('1. Cumulative returns...')
plot_cumulative({'DM': dm_full, 'EM': em_res, 'Commodities': cmd_res}, output_dir=FIG_DIR, oos_start=OOS_START)

# Fig 2: Sharpe bar charts
print('2. Sharpe ratio bars...')
plot_sharpe_bars({'DM': dm_full, 'EM': em_res, 'Commodities': cmd_res}, output_dir=FIG_DIR)

# Fig 3: Ablation heatmap
print('3. Ablation heatmap...')
if not ablation_df.empty:
    plot_ablation_heatmap(ablation_df, metric='Sharpe', output_dir=FIG_DIR)
    plot_ablation_heatmap(ablation_df, metric='Sortino', output_dir=FIG_DIR)

# Fig 4: Regime-conditional bars
print('4. Regime bars...')
if not regime_df.empty:
    plot_regime_bars(regime_df, metric='Sharpe', output_dir=FIG_DIR)
    plot_regime_bars(regime_df, metric='MaxDD', output_dir=FIG_DIR)

# Fig 5: Cost sensitivity curves
print('5. Cost sensitivity...')
if 'cost_df' in dir() and not cost_df.empty:
    plot_cost_sensitivity(cost_df, output_dir=FIG_DIR)

# Fig 6: Multi-seed boxplots
print('6. Seed robustness boxplots...')
if seed_stats_list:
    plot_seed_boxplots(seed_stats_list, metric='Sharpe', output_dir=FIG_DIR)
    plot_seed_boxplots(seed_stats_list, metric='Sortino', output_dir=FIG_DIR)

# Fig 7: Pairwise DM test heatmap
print('7. DM test heatmap...')
if 'dm_tests_sq' in dir() and not dm_tests_sq.empty:
    plot_pairwise_dm_heatmap(dm_tests_sq, output_dir=FIG_DIR)

# Table: Computational efficiency
# benchmark_efficiency expects {name: (model, is_torch)} tuples
print('\n8. Computational efficiency benchmark...')
models_dict = {
    'DHRP': (dhrp_dm, True),
    'MLP': (mlp_dm, True),
    'Transformer': (transformer_dm, True),
    'PPO': (ppo_dm, True),
}
if llm_dhrp_dm is not None:
    models_dict['LLM_DHRP'] = (llm_dhrp_dm, True)
n_assets_dm = DM_prices.shape[1]
fdim_dm = build_dataset(DM_prices)[0].shape[1]
eff_df = benchmark_efficiency(models_dict, n_assets=n_assets_dm, feature_dim=fdim_dm)
print(eff_df.to_string(index=False))
eff_df.to_csv('results/DM_computational_efficiency.csv', index=False)

print(f'\nAll figures saved to {FIG_DIR}/')
print('Files:')
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith('.png'):
        sz = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
        print(f'  {f} ({sz:.0f} KB)')

=== GENERATING PUBLICATION FIGURES ===

1. Cumulative returns...
2. Sharpe ratio bars...
3. Ablation heatmap...
4. Regime bars...
5. Cost sensitivity...
6. Seed robustness boxplots...
7. DM test heatmap...

8. Computational efficiency benchmark...
     Method  Params  Inference_ms
       DHRP   51432      2.155867
        MLP   55626      0.521020
Transformer  107521      1.012734
        PPO   88587      0.723467
   LLM_DHRP  177652      1.421261

All figures saved to results/figures/
Files:
  ablation_heatmap.png (83 KB)
  cost_sensitivity.png (159 KB)
  cumulative_returns.png (1175 KB)
  diagnostics_dm.png (72 KB)
  pairwise_dm_heatmap.png (172 KB)
  regime_bars.png (143 KB)
  seed_boxplots.png (95 KB)
  sharpe_bars.png (183 KB)


In [20]:
# === CELL 12: PAPER FIGURES ===
from src.visualization.plots import plot_cumulative, plot_sharpe_bars
import matplotlib.pyplot as plt
import pandas as pd

os.makedirs('results/figures', exist_ok=True)

results_dict = {'DM': dm_res, 'EM': em_res, 'Commodities': cmd_res}
plot_cumulative(results_dict, output_dir='results/figures', oos_start=OOS_START)
plot_sharpe_bars(results_dict, output_dir='results/figures')
print('Cumulative returns and Sharpe bar figures saved.')

# LLM-DHRP vs DHRP delta analysis
if 'LLM_DHRP' in dm_res['method'].unique():
    from src.visualization.plots import get_series
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (uname, res) in zip(axes, results_dict.items()):
        s_llm = get_series(res, 'LLM_DHRP')
        s_dhrp = get_series(res, 'DHRP')
        common = s_llm.index.intersection(s_dhrp.index)
        diff = s_llm.loc[common] - s_dhrp.loc[common]
        cum_diff = diff.cumsum() * 100
        ax.fill_between(common, cum_diff.values, 0,
                        where=cum_diff.values >= 0, alpha=0.3, color='green')
        ax.fill_between(common, cum_diff.values, 0,
                        where=cum_diff.values < 0, alpha=0.3, color='red')
        ax.plot(common, cum_diff.values, color='black', lw=1.5)
        ax.axhline(0, color='black', ls='--', lw=0.8)
        ax.axvline(pd.Timestamp(OOS_START), color='black', ls=':', lw=0.8, alpha=0.5)
        ax.set_title(f'{uname}: LLM-DHRP minus DHRP (OOS)', fontweight='bold')
        ax.set_ylabel('Cumulative Excess Return (%)')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/llm_delta.png', dpi=300)
    plt.show()
    print('LLM delta figure saved.')

Cumulative returns and Sharpe bar figures saved.
LLM delta figure saved.


In [ ]:
# === CELL 13: GATING INTERPRETABILITY ===
# Visualize how text features change the tree routing decisions
import matplotlib.pyplot as plt
import seaborn as sns

if llm_dhrp_dm is not None:
    from src.data.feature_engineering import build_dataset, make_features
    from src.data.llm_features import aggregate_text_per_timestep
    
    X, S, R, H = build_dataset(DM_prices)
    n_samples = min(50, X.shape[0])
    # Match training-time aggregation so text_dim aligns with model
    if text_dm is not None:
        text_aligned = aggregate_text_per_timestep(
            text_dm[:n_samples], method="norm_mean_max_concat"
        )
    
    # Compare gating with vs without text
    probs_with_text = []
    probs_without_text = []
    
    for i in range(n_samples):
        x = torch.from_numpy(X[i]).to(device)
        s = torch.from_numpy(S[i]).to(device)
        
        if text_dm is not None:
            te = torch.from_numpy(text_aligned[i]).to(device)
        else:
            te = torch.randn(llm_dhrp_dm.text_dim).to(device)  # auto-match dim
        
        p_with = llm_dhrp_dm.get_gating_probs(x, s, text_emb=te)
        p_without = llm_dhrp_dm.get_gating_probs(x, s, text_emb=None)
        probs_with_text.append([p.cpu().numpy() for p in p_with])
        probs_without_text.append([p.cpu().numpy() for p in p_without])
    
    # Plot root node gating shift
    root_with = [p[0][0] for p in probs_with_text]  # P(left) at root
    root_without = [p[0][0] for p in probs_without_text]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(root_without, label='Price-only', alpha=0.7, color='steelblue')
    ax.plot(root_with, label='Price + Text', alpha=0.7, color='crimson')
    ax.fill_between(range(n_samples),
                    [a-b for a, b in zip(root_with, root_without)],
                    alpha=0.2, color='crimson', label='Text impact')
    ax.set_xlabel('Sample')
    ax.set_ylabel('P(left) at root node')
    ax.set_title('Root Node Gating: Impact of Text Features', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/gating_interpretability.png', dpi=300)
    plt.show()
    print('Gating interpretability figure saved.')
else:
    print('LLM-DHRP not trained. Skipping interpretability analysis.')

In [ ]:
# === CELL 13.6: MULTI-UNIVERSE EXPANSION (Sectors, Global, Factors, Crypto, Bonds) ===
# For NeurIPS 2026 D&B + ICLR 2027: empirical breadth across 8 universes total
from src.data.price_loader import load_universe, load_etf_volume_data, UNIVERSES
from src.training.trainer import train_dhrp, train_llm_dhrp_warmstart
from src.evaluation.backtest import rolling_backtest
from src.evaluation.statistics import compute_stats
from datetime import datetime, timedelta
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

NEW_UNIVERSES = ['Sectors', 'Global', 'Factors', 'Crypto', 'Bonds']
all_results = {}
all_stats = []

for u_name in NEW_UNIVERSES:
    print(f'\n{"="*60}\n  {u_name.upper()} UNIVERSE\n{"="*60}')
    try:
        # Crypto has limited history, use shorter start
        u_start = '2020-07-01' if u_name == 'Crypto' else START
        prices = load_universe(u_name, u_start, END)
        if prices.empty or prices.shape[1] < 4:
            print(f'  Skipping {u_name}: insufficient data')
            continue
        print(f'  Loaded {prices.shape[0]} rows x {prices.shape[1]} assets')

        # Volume (best-effort)
        try:
            vol = load_etf_volume_data(list(UNIVERSES[u_name].values()), u_start, END)
        except Exception:
            vol = None

        # Train DHRP
        u_train_end = '2022-12-31' if u_name == 'Crypto' else TRAIN_END
        try:
            dhrp_u = train_dhrp(prices, device=device, is_em=False, volume=vol, train_end=u_train_end)
        except Exception as e:
            print(f'  DHRP training failed: {e}')
            continue

        # Backtest (no LLM-DHRP for new universes — fast track for D&B paper)
        u_oos = '2023-01-01' if u_name == 'Crypto' else OOS_START
        methods = ['EW', 'MINVAR', 'MV', 'HRP', 'RP', 'MAXDIV', 'DHRP']
        res = rolling_backtest(
            prices, is_em=False, dhrp_model=dhrp_u,
            methods=methods, volume=vol,
            oos_start=u_oos, universe=u_name, weight_ema=0.3,
        )
        if res.empty:
            print(f'  Backtest empty')
            continue

        all_results[u_name] = res
        stats = compute_stats(res, oos_start=u_oos)
        stats['Universe'] = u_name
        all_stats.append(stats)
        print(f'  Top 3 by Sharpe:')
        print(stats.sort_values('Sharpe', ascending=False).head(3)[['Method', 'Sharpe', 'HAC_t', 'MaxDD']].to_string(index=False))

        # Save
        stats.to_csv(f'results/{u_name}_results.csv', index=False)
        torch.save(dhrp_u.state_dict(), f'results/models/dhrp_{u_name.lower()}.pt')
    except Exception as e:
        print(f'  {u_name} failed: {type(e).__name__}: {e}')

# Aggregate summary across all 8 universes
print(f'\n{"="*60}\n  ALL-UNIVERSE SUMMARY\n{"="*60}')
if all_stats:
    summary = pd.concat(all_stats, ignore_index=True)
    summary = summary[['Universe', 'Method', 'Sharpe', 'HAC_t', 'MaxDD', 'Calmar']]
    summary.to_csv('results/all_new_universes_summary.csv', index=False)
    # Pivot: rows=Method, cols=Universe, values=Sharpe
    pivot = summary.pivot(index='Method', columns='Universe', values='Sharpe').round(3)
    print('\nSharpe by Method x Universe:')
    print(pivot.to_string())
    pivot.to_csv('results/sharpe_pivot_new_universes.csv')
print('Multi-universe expansion complete.')

In [ ]:
# === CELL 13.5: PROBE ANALYSIS (interpretability) ===
# Tests whether the DHRP tree gates learn regime-discriminative features.
# Required for top-venue interpretability section (ICLR/ICML/NeurIPS).
from scripts.probe_analysis import run_full_probe
from src.data.feature_engineering import build_dataset
import pandas as pd
import numpy as np

# Asset category ground truth for purity analysis
dm_categories = {
    'SPY': 'equity', 'QQQ': 'equity', 'IWM': 'equity', 'EFA': 'equity', 'VGK': 'equity',
    'TLT': 'bond', 'IEF': 'bond', 'LQD': 'bond',
    'VNQ': 'alternative', 'UUP': 'currency',
}

# Build FULL dataset (no train_end) so probe covers all market regimes:
# pre_covid_bull, covid_crash, recovery, rate_hikes, post_hike
X_dm, S_dm, R_dm, _ = build_dataset(DM_prices, volume=DM_vol)
dm_rets = DM_prices.pct_change().dropna()
# Sample dates: build_dataset steps by 5 days from window=252
sample_dates = dm_rets.index[252::5][:len(X_dm)]
print(f'Probe samples: {len(X_dm)}, date range: {sample_dates[0].date()} to {sample_dates[-1].date()}')

print('\n=== PROBE: DHRP (DM, price-only) ===')
probe_dhrp_dm = run_full_probe(
    dhrp_dm, X_dm, S_dm, list(sample_dates),
    asset_names=list(DM_prices.columns),
    returns=dm_rets,
    asset_categories=dm_categories,
    device=device, label='DHRP_DM',
)

if llm_dhrp_dm is not None and text_dm is not None:
    # Reconstruct text embeddings aligned to X_dm (full period)
    from src.data.llm_features import aggregate_text_per_timestep
    n_text = min(text_dm.shape[0], X_dm.shape[0])
    # Match training-time aggregation (norm_mean_max_concat -> 1536 dim)
    text_embs_probe = aggregate_text_per_timestep(
        text_dm[:n_text], method="norm_mean_max_concat"
    )
    if n_text < X_dm.shape[0]:
        text_embs_probe = np.vstack([text_embs_probe,
            np.zeros((X_dm.shape[0] - n_text, text_embs_probe.shape[1]), dtype=np.float32)])

    print('\n=== PROBE: LLM-DHRP (DM, price+text) ===')
    probe_llm_dm = run_full_probe(
        llm_dhrp_dm, X_dm, S_dm, list(sample_dates),
        asset_names=list(DM_prices.columns),
        returns=dm_rets,
        asset_categories=dm_categories,
        text_embs=text_embs_probe,
        device=device, label='LLM_DHRP_DM',
    )

    # Compare: does text improve regime discrimination?
    print('\n=== REGIME PROBE COMPARISON ===')
    print(f'  DHRP test acc:     {probe_dhrp_dm["regime_probe"]["test_acc"]:.3f}')
    print(f'  LLM-DHRP test acc: {probe_llm_dm["regime_probe"]["test_acc"]:.3f}')
    lift = probe_llm_dm['regime_probe']['test_acc'] - probe_dhrp_dm['regime_probe']['test_acc']
    print(f'  LLM regime lift:   {lift:+.3f}')

print('\nProbe analysis complete. Figures in results/figures/probe_gates_*.png')

In [23]:
# === CELL 14: FINAL SUMMARY ===
print('=' * 70)
print('  DHRP EXPERIMENT COMPLETE')
print('=' * 70)

print(f'Universes tested: DM ({DM_prices.shape[1]}), EM ({EM_prices.shape[1]}), CMD ({CMD_prices.shape[1]})')
print(f'Total assets: {DM_prices.shape[1] + EM_prices.shape[1] + CMD_prices.shape[1]}')
methods_str = sorted(dm_full['method'].unique())
print(f'Methods compared: {methods_str}')
print(f'Training period: {START} to {TRAIN_END}')
print(f'OOS test period: {OOS_START} to {END}')
print(f'Device used: {device}')

print()
print('--- Headline DHRP results (FF6 alpha, Newey-West HAC) ---')
for u, df in [('DM', dm_full_stats), ('EM', em_res), ('CMD', cmd_res)]:
    try:
        if hasattr(df, 'set_index'):
            row = df.set_index('Method').loc['DHRP'] if 'Method' in df.columns else None
            if row is not None:
                print(f'  {u}: Sharpe={row["Sharpe"]:.3f}')
    except Exception as e:
        pass
    

print()
print('--- Result CSVs ---')
import glob
for f in sorted(glob.glob('results/*.csv')):
    print(f'  {f}')

print()
print('--- Figures ---')
for f in sorted(glob.glob('results/figures/*.png')):
    sz = os.path.getsize(f) / 1024
    print(f'  {f} ({sz:.0f} KB)')

print()
print('--- Benchmarks Completed ---')
print('  1. Deep baselines (MLP, Transformer, PPO)')
print('  2. Full OOS backtest (10+ methods, 3 universes)')
print('  3. Pairwise statistical tests (Sharpe + DM, Holm-Bonferroni)')
print('  4. Superior Predictive Ability (SPA) test')
print('  5. Model Confidence Set (MCS)')
print('  6. Ablation studies (tree depth, loss components)')
print('  7. Multi-seed robustness (5 seeds)')
print('  8. Transaction cost sensitivity + turnover')
print('  9. Regime-conditional analysis (5 macro regimes)')
print('  10. Publication figures (NeurIPS-compatible, 300 DPI)')
print('  11. Ledoit-Wolf shrinkage covariance estimation')
print('  12. Strict OOS train/test split (no look-ahead bias)')
print('  13. LLM-DHRP brief negative ablation (mentioned only)')

if torch.cuda.is_available():
    print(f'GPU memory peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')


  DHRP EXPERIMENT COMPLETE
Universes tested: DM (10), EM (10), CMD (10)
Total assets: 30
Methods compared: ['DFL', 'DHRP', 'EW', 'HRP', 'LLM_DHRP', 'MAXDIV', 'MINVAR', 'MLP', 'MV', 'PPO', 'RP', 'Transformer']
Training period: 2012-04-22 to 2020-06-30
OOS test period: 2020-07-01 to 2026-04-19
Device used: cuda

--- Headline DHRP results (FF6 alpha, Newey-West HAC) ---
  DM: Sharpe=0.133

--- Result CSVs ---
  results/Commodities_dm_tests_negative.csv
  results/Commodities_dm_tests_squared.csv
  results/Commodities_regime_analysis.csv
  results/Commodities_results.csv
  results/Commodities_sharpe_tests.csv
  results/DM_ablations.csv
  results/DM_computational_efficiency.csv
  results/DM_cost_sensitivity.csv
  results/DM_dm_tests_negative.csv
  results/DM_dm_tests_squared.csv
  results/DM_full_stats.csv
  results/DM_multiseed.csv
  results/DM_regime_analysis.csv
  results/DM_results.csv
  results/DM_sharpe_tests.csv
  results/DM_turnover.csv
  results/EM_dm_tests_negative.csv
  results/EM